In [17]:
!pip install -q pypdf sentence-transformers chromadb openai

In [18]:
from google.colab import userdata, files
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
import os

In [20]:
# Get OpenRouter API key from Colab Secrets
OPENROUTER_API_KEY = userdata.get("PDF_Chatbot")

# Check whether key was loaded
if not OPENROUTER_API_KEY:
    raise ValueError("API key not found. Please create a Colab Secret named 'PDF_Chatbot'.")

print("API key loaded successfully!")

API key loaded successfully!


In [21]:
# OpenRouter uses an OpenAI-compatible API
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Gemini model through OpenRouter
MODEL_NAME = "google/gemini-2.5-flash"

print("OpenRouter client initialized successfully!")

OpenRouter client initialized successfully!


In [22]:
print("Please upload your PDF file:")

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print("PDF uploaded successfully:")
print(pdf_filename)

Please upload your PDF file:


Saving DF_48,27.pdf to DF_48,27 (1).pdf
PDF uploaded successfully:
DF_48,27 (1).pdf


In [23]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)

pdf_text = ""

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        pdf_text += text + "\n"

print("PDF text extracted successfully!")
print("Number of characters:", len(pdf_text))

PDF text extracted successfully!
Number of characters: 5664


In [ ]:
print(pdf_text[:2000])

In [24]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [25]:
chunks = chunk_text(pdf_text)

print("Total PDF chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print("\n--- CHUNK", i + 1, "---")
    print(chunk[:500])

Total PDF chunks: 8

--- CHUNK 1 ---
DF
 
 
 
 
 
 
 
 
 
 
 
Roll
 
No
 
48
 
&
 
27
 
Tutorial
 
2
 
Case
 
Study:Fake
 
Social
 
Media
 
Account
 
Investigation
 
1.
 
Case
 
Background
 
○
 
Brief
 
description
 
of
 
the
 
complaint.
 
○
 
A
 
fake
 
social-media
 
account
 
was
 
allegedly
 
created
 
using
 
the
 
victim’s
 
name/photo.
 
○
 
Explain
 
the
 
suspected
 
activity,
 
such
 
as
 
impersonation,
 
harassment,
 
threats,
 
or
 
fraudulent
 
communication.
 
○
 
Mention
 
the
 
date,
 
investigating
 
officer,
 
c

--- CHUNK 2 ---
a
 
profile.
 
○
 
E-03:
 
Screenshots
 
of
 
relevant
 
messages/posts.
 
○
 
E-04:
 
Account/profile
 
information
 
supplied
 
during
 
the
 
investigation.
 
○
 
E-05:
 
Exported
 
digital
 
files
 
or
 
photographs,
 
if
 
applicable.
 
 
3.
 
Evidence
 
ID
 
for
 
Each
 
Item
 
 
 
Assign
 
a
 
unique
 
identifier
 
to
 
every
 
item:
 
 
Evidence
 
ID
 
Evidence
 
Item
 
E-01
 
Mobile
 
phone
 
E-02
 
Fake-account
 
profile
 
screensh

In [26]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [27]:
embeddings = embedder.encode(chunks).tolist()

print("Embeddings created successfully!")
print("Number of embeddings:", len(embeddings))

Embeddings created successfully!
Number of embeddings: 8


In [28]:
# Create ChromaDB client
chroma_client = chromadb.Client()

# Create collection
collection = chroma_client.get_or_create_collection(
    name="pdf_chatbot"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [29]:
# Create unique IDs
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print("PDF chunks stored in ChromaDB successfully!")
print("Total documents in collection:", collection.count())

PDF chunks stored in ChromaDB successfully!
Total documents in collection: 8


In [31]:
def retrieve_pdf_context(query: str, top_k: int = 3) -> list[str]:

    # Convert user's question into embedding
    query_embedding = embedder.encode([query]).tolist()

    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    # Return retrieved PDF documents
    return results["documents"][0]

In [32]:
def ask_pdf(query: str):

    # Retrieve relevant PDF passages
    context_passages = retrieve_pdf_context(query, top_k=3)

    # Combine retrieved passages
    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    # Create prompt
    prompt = f"""
You are an intelligent document analysis assistant.

Answer the question using ONLY the information provided
in the PDF context below.

If the information is not contained within the provided
context, clearly state:

"I cannot find the answer in the provided PDF."

Do not make up information.

PDF Context:
{context_str}

Question:
{query}

Answer:
"""

    # Send request to OpenRouter
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    # Get answer
    answer = response.choices[0].message.content

    return answer, context_passages

In [34]:
def ask_pdf(query: str):

    # Retrieve relevant PDF passages
    context_passages = retrieve_pdf_context(query, top_k=3)

    # Combine retrieved passages
    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    # Prompt
    prompt = f"""
You are an intelligent PDF document analysis assistant.

Answer the user's question using ONLY the information
provided in the PDF context below.

If the information is not available in the PDF context,
say:
"I cannot find the answer in the provided PDF."

For summary questions, give a clear and concise summary
based on the retrieved PDF content.

Do not make up information.

PDF Context:
{context_str}

Question:
{query}

Answer:
"""

    # OpenRouter API call
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1000       # IMPORTANT: prevents 65535-token request
    )

    answer = response.choices[0].message.content

    return answer, context_passages

In [36]:
question = input("Ask a question about your PDF: ")

answer, context = ask_pdf(question)

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)

print(answer)

Ask a question about your PDF: What is this PDF about? Give me a summary of it.

ANSWER
This PDF outlines a case study for a fake social media account investigation. It details the background of the complaint, including the alleged creation of a fake account using a victim's name/photo and suspected activities like impersonation or harassment. The document also lists digital evidence items, assigns unique identifiers to each, and describes the evidence, such as seized mobile phones, screenshots of the fake profile and messages, and account information. It also covers packaging and sealing details for evidence and provides a transfer history for two cases (Nandini Rai and Riki Das).


In [37]:
print("\n" + "=" * 60)
print("RETRIEVED PDF SNIPPETS")
print("=" * 60)

for i, snippet in enumerate(context, 1):
    print(f"\n--- Snippet {i} ---")
    print(snippet[:1500])


RETRIEVED PDF SNIPPETS

--- Snippet 1 ---
DF
 
 
 
 
 
 
 
 
 
 
 
Roll
 
No
 
48
 
&
 
27
 
Tutorial
 
2
 
Case
 
Study:Fake
 
Social
 
Media
 
Account
 
Investigation
 
1.
 
Case
 
Background
 
○
 
Brief
 
description
 
of
 
the
 
complaint.
 
○
 
A
 
fake
 
social-media
 
account
 
was
 
allegedly
 
created
 
using
 
the
 
victim’s
 
name/photo.
 
○
 
Explain
 
the
 
suspected
 
activity,
 
such
 
as
 
impersonation,
 
harassment,
 
threats,
 
or
 
fraudulent
 
communication.
 
○
 
Mention
 
the
 
date,
 
investigating
 
officer,
 
case/reference
 
number,
 
and
 
location
 
as
 
appropriate.
 
○
 
State
 
the
 
objective
 
of
 
the
 
forensic
 
investigation.
 
2.
 
List
 
of
 
Digital
 
Evidence
 
 
 
Example
 
evidence
 
items:
 
 
○
 
E-01:
 
Seized
 
mobile
 
phone.
 
○
 
E-02:
 
Screenshots
 
of
 
the
 
fake
 
social-media
 
profile.
 
○
 
E-03:
 
Screenshots
 
of
 
relevant
 
messages/posts.
 
○
 
E-04:
 
Account/profile
 
information
 
supplied
 
during
 
the
 
investigatio

In [ ]:
print("\n" + "=" * 60)
print("          PDF CHATBOT READY!")
print("=" * 60)

print("Ask questions about your PDF.")
print("Type 'exit', 'quit' or 'q' to stop.")
print("=" * 60)


while True:

    user_query = input("\nAsk a question about your PDF: ")

    # Exit chatbot
    if user_query.lower() in ["exit", "quit", "q"]:
        print("\nExiting PDF Chatbot. Goodbye!")
        break

    # Ignore empty input
    if not user_query.strip():
        continue

    # Get answer from PDF
    answer, context = ask_pdf(user_query)

    # --------------------------------------------------------
    # RETRIEVED PDF SNIPPETS
    # --------------------------------------------------------

    print("\n" + "-" * 60)
    print("RETRIEVED PDF SNIPPETS")
    print("-" * 60)

    for i, snippet in enumerate(context, 1):
        print(f"\n[{i}] {snippet[:1500]}")

    # --------------------------------------------------------
    # CHATBOT RESPONSE
    # --------------------------------------------------------

    print("\n" + "-" * 60)
    print("CHATBOT RESPONSE")
    print("-" * 60)

    print(answer)

    print("-" * 60)


          PDF CHATBOT READY!
Ask questions about your PDF.
Type 'exit', 'quit' or 'q' to stop.

Ask a question about your PDF: Give summary of this pdf in 200 words

------------------------------------------------------------
RETRIEVED PDF SNIPPETS
------------------------------------------------------------

[1] a
 
profile.
 
○
 
E-03:
 
Screenshots
 
of
 
relevant
 
messages/posts.
 
○
 
E-04:
 
Account/profile
 
information
 
supplied
 
during
 
the
 
investigation.
 
○
 
E-05:
 
Exported
 
digital
 
files
 
or
 
photographs,
 
if
 
applicable.
 
 
3.
 
Evidence
 
ID
 
for
 
Each
 
Item
 
 
 
Assign
 
a
 
unique
 
identifier
 
to
 
every
 
item:
 
 
Evidence
 
ID
 
Evidence
 
Item
 
E-01
 
Mobile
 
phone
 
E-02
 
Fake-account
 
profile
 
screenshots
 
E-03
 
Message/post
 
screenshots
 
E-04
 
Account
 
information
 
E-05
 
Related
 
exported
 
digital
 
files
 
 
DF
 
 
 
 
 
 
 
 
 
 
 
Roll
 
No
 
48
 
&
 
27
 
 
 
4.
 
Evidence
 
Description
 
 
 
Record
 
objective
 
details